# Silver Layer — ICD Procedure Dictionary (`d_icd_procedures`)

**Source -> Target:** `<source_schema>.d_icd_procedures` -> `<target_schema>.d_icd_procedures` (defaults `cdr_mimiciv.bronze` -> `cdr_mimiciv.silver`). The target is replaced via `CREATE OR REPLACE`; downstream Gold depends on it.

**Objective:** Standardize the ICD procedure reference dictionary while preserving ICD-9 and ICD-10 distinctions and original procedure descriptions.

**Expected grain:** One row per ICD procedure code and ICD version (`icd_code`, `icd_version`; both NOT NULL per MIMIC-IV `hosp.d_icd_procedures`, which `procedures_icd` links to on those two columns).

**Silver rules:** Preserve all source records and clinical meaning, standardize data types and whitespace, validate uniqueness and completeness, and reconcile Bronze–Silver counts. No imputation, automatic deduplication or analytical aggregation.

In [0]:
# Import Spark functions
from pyspark.sql import functions as F

In [0]:
# Configure Bronze source and Silver target schemas (widgets allow standalone and dbutils.notebook.run use)
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

TABLE = "d_icd_procedures"
SOURCE_TABLE = f"{BRONZE}.{TABLE}"
TARGET_TABLE = f"{SILVER}.{TABLE}"

# Expected published contract (ordered column -> type)
EXPECTED_SCHEMA = {
    "icd_code": "string",
    "icd_version": "int",
    "long_title": "string",
}

In [0]:
# Read and inspect the ICD procedure dictionary
bronze_df = spark.table(SOURCE_TABLE)

bronze_df.printSchema()
display(bronze_df.limit(20))

print("Bronze rows:", bronze_df.count())

root
 |-- icd_code: string (nullable = true)
 |-- icd_version: integer (nullable = true)
 |-- long_title: string (nullable = true)



icd_code,icd_version,long_title
0001,9,Therapeutic ultrasound of vessels of head and neck
0002,9,Therapeutic ultrasound of heart
0003,9,Therapeutic ultrasound of peripheral vascular vessels
0009,9,Other therapeutic ultrasound
001,10,"Central Nervous System and Cranial Nerves, Bypass"
0010,9,Implantation of chemotherapeutic agent
0011,9,Infusion of drotrecogin alfa (activated)
0012,9,Administration of inhaled nitric oxide
0013,9,Injection or infusion of nesiritide
0014,9,Injection or infusion of oxazolidinone class of antibiotics


Bronze rows: 86423


In [0]:
# Profile source records and missing values
bronze_profile = bronze_df.select(
    F.count("*").alias("total_rows"),
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
        for c in ["icd_code", "icd_version", "long_title"]
    ]
)

display(bronze_profile)

total_rows,null_icd_code,null_icd_version,null_long_title
86423,0,0,0


In [0]:
# Standardize procedure codes, ICD versions and descriptions
silver_df = bronze_df.select(
    F.trim(F.col("icd_code").cast("string")).alias("icd_code"),
    F.col("icd_version").cast("int").alias("icd_version"),
    F.trim(F.col("long_title").cast("string")).alias("long_title")
)

display(silver_df.limit(20))
silver_df.printSchema()

icd_code,icd_version,long_title
0001,9,Therapeutic ultrasound of vessels of head and neck
0002,9,Therapeutic ultrasound of heart
0003,9,Therapeutic ultrasound of peripheral vascular vessels
0009,9,Other therapeutic ultrasound
001,10,"Central Nervous System and Cranial Nerves, Bypass"
0010,9,Implantation of chemotherapeutic agent
0011,9,Infusion of drotrecogin alfa (activated)
0012,9,Administration of inhaled nitric oxide
0013,9,Injection or infusion of nesiritide
0014,9,Injection or infusion of oxazolidinone class of antibiotics


root
 |-- icd_code: string (nullable = true)
 |-- icd_version: integer (nullable = true)
 |-- long_title: string (nullable = true)



In [0]:
# Check uniqueness of ICD procedure code-version combinations
duplicate_df = (
    silver_df
    .groupBy("icd_code", "icd_version")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_df.count()

print("Duplicate ICD procedure keys:", duplicate_count)
display(duplicate_df)

Duplicate ICD procedure keys: 0


icd_code,icd_version,count


In [0]:
# Check missing procedure codes, versions and descriptions
quality_df = silver_df.select(
    F.sum(
        F.when(
            F.col("icd_code").isNull() |
            (F.col("icd_code") == ""), 1
        ).otherwise(0)
    ).alias("missing_icd_codes"),

    F.sum(
        F.col("icd_version").isNull().cast("int")
    ).alias("null_icd_versions"),

    F.sum(
        F.when(
            F.col("icd_version").isNotNull() &
            (~F.col("icd_version").isin(9, 10)), 1
        ).otherwise(0)
    ).alias("invalid_icd_versions"),

    F.sum(
        F.when(
            F.col("long_title").isNull() |
            (F.col("long_title") == ""), 1
        ).otherwise(0)
    ).alias("missing_descriptions")
)

display(quality_df)

missing_icd_codes,null_icd_versions,invalid_icd_versions,missing_descriptions
0,0,0,0


In [0]:
# Verify that the Silver transformation preserves all Bronze rows
bronze_count = bronze_df.count()
silver_count = silver_df.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)

assert bronze_count == silver_count, "Bronze–Silver row-count mismatch"

Bronze rows: 86423
Silver rows: 86423


In [0]:
# Stop publication if critical procedure dictionary checks fail
quality = quality_df.first().asDict()

assert duplicate_count == 0, f"Duplicate procedure keys: {duplicate_count}"
assert quality["missing_icd_codes"] == 0, "Missing procedure codes"
assert quality["null_icd_versions"] == 0, "Missing ICD versions"
assert quality["invalid_icd_versions"] == 0, "Invalid ICD versions"
assert quality["missing_descriptions"] == 0, "Missing procedure descriptions"
assert bronze_count == silver_count, "Record counts do not match"

print("All critical Silver quality checks passed.")

All critical Silver quality checks passed.


In [0]:
# Publish the validated Silver ICD procedure dictionary (CREATE OR REPLACE; downstream Gold depends on it)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

silver_df.writeTo(TARGET_TABLE).using("delta").createOrReplace()

print("Published:", TARGET_TABLE)

Published: cdr_mimiciv.silver.d_icd_procedures


In [0]:
# Verify the published table and record counts
published_df = spark.table(TARGET_TABLE)

published_count = published_df.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Published rows:", published_count)

assert published_count == silver_count, "Published row-count mismatch"
published_schema = {f.name: f.dataType.simpleString() for f in published_df.schema.fields}
assert list(published_schema.items()) == list(EXPECTED_SCHEMA.items()), (
    f"Published schema differs from contract: {published_schema}"
)

published_keys = published_df.select(
    F.sum(
        (F.col("icd_code").isNull() | F.col("icd_version").isNull()).cast("int")
    ).alias("null_keys"),
    F.countDistinct("icd_code", "icd_version").alias("distinct_keys")
).first()
assert published_keys["null_keys"] == 0, "Published key columns contain NULLs"
assert published_keys["distinct_keys"] == published_count, "Published keys are not unique"

display(published_df.limit(20))

print("Silver d_icd_procedures table verified successfully.")

Bronze rows: 86423
Silver rows: 86423
Published rows: 86423


icd_code,icd_version,long_title
0001,9,Therapeutic ultrasound of vessels of head and neck
0002,9,Therapeutic ultrasound of heart
0003,9,Therapeutic ultrasound of peripheral vascular vessels
0009,9,Other therapeutic ultrasound
001,10,"Central Nervous System and Cranial Nerves, Bypass"
0010,9,Implantation of chemotherapeutic agent
0011,9,Infusion of drotrecogin alfa (activated)
0012,9,Administration of inhaled nitric oxide
0013,9,Injection or infusion of nesiritide
0014,9,Injection or infusion of oxazolidinone class of antibiotics


Silver d_icd_procedures table verified successfully.


In [0]:
# Finish successfully only after every assertion above has passed
dbutils.notebook.exit(f"{TABLE}:OK")